# 7.27 OCR and scene text

OCR is visual sequence modeling: find text-shaped evidence in the image, then align a variable-width strip of features to characters.

CTC lets many frame-level paths collapse to one string, with blanks acting as alignment glue for repeated letters and variable spacing.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
np.set_printoptions(precision=3, suppress=True)

## The concept, built once

CTC sums path probabilities:

$$P(y\mid x)=\sum_{\pi\in\mathcal B^{-1}(y)}\prod_t p_t(\pi_t\mid x).$$

We verify sequence length $25$, `bok` vs `book`, and total probability $0.40$.

In [ ]:
blank = "_"


def ctc_collapse(path):
    merged = []
    last = None
    for token in path:
        if token != last:
            merged.append(token)
        last = token
    return "".join(token for token in merged if token != blank)


def wrong_collapse(path):
    no_blank = [token for token in path if token != blank]
    merged = []
    last = None
    for token in no_blank:
        if token != last:
            merged.append(token)
        last = token
    return "".join(merged)


def ocr_ctc_decode(frame_tokens):
    return ctc_collapse(frame_tokens)


image_width = 100
stride = 4
sequence_length = image_width // stride
bad_path = ["b", blank, "o", "o", blank, "k"]
good_path = ["b", blank, "o", blank, "o", "k"]
probability = 0.8 * 0.5 * 0.7 + 0.12
assert sequence_length == 25
assert ctc_collapse(bad_path) == "bok"
assert ctc_collapse(good_path) == "book"
assert round(probability, 2) == 0.40
print("sequence length", sequence_length)
print("bad path", ctc_collapse(bad_path))
print("good path", ctc_collapse(good_path))
print("string probability", round(probability, 2))

## Dataset ladder

We synthesize word images and matching frame-token paths with increasing length, repeated letters, noise, and blank ambiguity. The metric is character accuracy after CTC decoding.

In [ ]:
glyphs = {
    "b": np.array([[1, 1, 0], [1, 0, 1], [1, 1, 0], [1, 0, 1], [1, 1, 0]], dtype=float),
    "o": np.array([[0, 1, 0], [1, 0, 1], [1, 0, 1], [1, 0, 1], [0, 1, 0]], dtype=float),
    "k": np.array([[1, 0, 1], [1, 1, 0], [1, 0, 0], [1, 1, 0], [1, 0, 1]], dtype=float),
    "c": np.array([[0, 1, 1], [1, 0, 0], [1, 0, 0], [1, 0, 0], [0, 1, 1]], dtype=float),
    "t": np.array([[1, 1, 1], [0, 1, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0]], dtype=float),
    "a": np.array([[0, 1, 0], [1, 0, 1], [1, 1, 1], [1, 0, 1], [1, 0, 1]], dtype=float),
    "r": np.array([[1, 1, 0], [1, 0, 1], [1, 1, 0], [1, 1, 0], [1, 0, 1]], dtype=float),
}


def render_word(word, noise, seed):
    local_rng = np.random.default_rng(seed)
    columns = []
    for char in word:
        columns.append(glyphs[char])
        columns.append(np.zeros((5, 1)))
    image = np.concatenate(columns, axis=1)
    image = image + local_rng.normal(0.0, noise, size=image.shape)
    return np.clip(image, 0.0, 1.0)


def path_for_word(word, confuse_repeat=False):
    path = []
    for char in word:
        path.append(char)
        if confuse_repeat and len(path) > 1 and path[-2] == char:
            path.append(char)
        else:
            path.append(blank)
    if path:
        path.pop()
    return path


def char_accuracy(pred, target):
    n = max(len(pred), len(target))
    matches = sum(a == b for a, b in zip(pred, target))
    return matches / n


def make_ocr_ladder():
    specs = [
        ("D1 clean book", "book", 0.00, False),
        ("D2 short cat", "cat", 0.02, False),
        ("D3 repeated book", "book", 0.04, False),
        ("D4 noisy track", "track", 0.08, False),
        ("D5 blank risk", "book", 0.12, True),
    ]
    rungs = []
    for seed, spec in enumerate(specs):
        name, word, noise, confuse = spec
        image = render_word(word, noise, seed + 60)
        path = path_for_word(word, confuse_repeat=confuse)
        if confuse:
            path = ["b", blank, "o", "o", blank, "k"]
        rungs.append({"name": name, "word": word, "image": image, "path": path})
    return rungs


ocr_rungs = make_ocr_ladder()
for rung in ocr_rungs:
    print(rung["name"], "target", rung["word"], "frames", len(rung["path"]), "image", rung["image"].shape)

fig, axes = plt.subplots(1, 5, figsize=(12, 2.4))
for ax, rung in zip(axes, ocr_rungs):
    ax.imshow(rung["image"], cmap="gray", vmin=0, vmax=1)
    ax.set_title(rung["name"])
    ax.axis("off")
plt.tight_layout()

In [ ]:
ocr_results = []
for rung in ocr_rungs:
    pred = ocr_ctc_decode(rung["path"])
    acc = char_accuracy(pred, rung["word"])
    ocr_results.append({"name": rung["name"], "pred": pred, "acc": acc})

print("rung                 pred    target  char acc")
for rung, result in zip(ocr_rungs, ocr_results):
    print(f"{rung['name']:<20} {result['pred']:<7} {rung['word']:<7} {result['acc']:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, rung in enumerate(ocr_rungs):
    axes[0, i].imshow(rung["image"], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(ocr_results[i]["pred"])
    axes[0, i].axis("off")

scores = [item["acc"] for item in ocr_results]
axes[1, 0].plot(range(1, 6), scores, marker="o")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("character accuracy")
axes[1, 0].set_title("CTC decode quality")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: deleting blanks before merging repeats

For repeated letters, the order of CTC collapse operations decides whether `book` survives as two `o` characters.

In [ ]:
path = ["b", blank, "o", blank, "o", "k"]
wrong = wrong_collapse(path)
right = ctc_collapse(path)
small_product = float(np.prod([0.01] * 80))
log_product = float(np.sum(np.log([0.01] * 80)))
print("wrong collapse", wrong)
print("right collapse", right)
print("tiny product", small_product)
print("log-space score", round(log_product, 2))

## Evaluate it + Practice

- Metric: character accuracy after CTC collapse; no-skill baseline is an empty string.
- Sanity check: `book` needs a blank between the two `o` labels.
- Ablation: use `wrong_collapse` and repeated letters fail.
- Failure signal: probabilities underflow when long paths are multiplied outside log space.

Practice:
1. Add a new glyph.
2. Make D5 choose the wrong frame token.
3. Implement edit distance instead of position accuracy.